# Indian Housing Price Prediction
### Specialized Model handling BHK, Area, City, and Furnishing

In [1]:
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score


In [2]:
train = pd.read_csv('data/indian_train.csv')
test = pd.read_csv('data/indian_test.csv')
print('Train Shape:', train.shape)
print('Test Shape:', test.shape)


Train Shape: (2500, 9)
Test Shape: (500, 8)


In [3]:
X = train.drop(columns=['Property_ID', 'Price_Lakhs'])
y = train['Price_Lakhs']
X_test = test.drop(columns=['Property_ID'])

numerical_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, numerical_cols),
    ('cat', cat_pipeline, categorical_cols)
])


In [4]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', Ridge(alpha=1.0))
])
model_pipeline.fit(X_train, y_train)
preds = model_pipeline.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, preds))
r2 = r2_score(y_val, preds)
print(f"Validation RMSE: {rmse:.2f} Lakhs")
print(f"R2 Score: {r2:.4f}")


Validation RMSE: 47.04 Lakhs
R2 Score: 0.8859


In [5]:
with open('indian_house_price_model.pkl', 'wb') as f:
    pickle.dump(model_pipeline, f)
print('Model saved to indian_house_price_model.pkl')


Model saved to indian_house_price_model.pkl


In [6]:
test_preds = model_pipeline.predict(X_test)
submission = pd.DataFrame({'Property_ID': test['Property_ID'], 'Price_Lakhs': test_preds})
submission.to_csv('indian_submission.csv', index=False)
print('Predictions saved to indian_submission.csv')


Predictions saved to indian_submission.csv


## Interactive Live Prediction (Demo!)
Run this block to manually type in properties and receive a live estimated House Price!

In [7]:
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

with open('indian_house_price_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

reference_columns = ['City', 'Area_SqFt', 'BHK', 'Property_Age_Years', 'Furnishing', 'Parking', 'RERA_Approved']
custom_house = pd.DataFrame(columns=reference_columns)
custom_house.loc[0] = np.nan

print("=========================================")
print("    🏠 ENTER CUSTOM HOUSE FEATURES 🏠    ")
print("=========================================")
try:
    print("\nCity Options: [1] Mumbai, [2] Delhi, [3] Bangalore, [4] Hyderabad, [5] Chennai, [6] Pune, [7] Ahmedabad")
    c = input("Enter City Option (1-7)? [Press Enter for 3:Bangalore]: ").strip()
    city_map = {'1':'Mumbai', '2':'Delhi', '3':'Bangalore', '4':'Hyderabad', '5':'Chennai', '6':'Pune', '7':'Ahmedabad'}
    city = city_map.get(c, 'Bangalore')

    a = input("\nArea (in SqFt)? [Press Enter for 1200]: ").strip()
    area = int(a) if a else 1200

    b = input("\nBHK (1-6)? [Press Enter for 3]: ").strip()
    bhk = int(b) if b else 3
    
    print("\nFurnishing Options: [1] Unfurnished, [2] Semi-Furnished, [3] Fully-Furnished")
    f = input("Enter Furnishing (1-3)? [Press Enter for 3:Fully-Furnished]: ").strip()
    furn_map = {'1':'Unfurnished', '2':'Semi-Furnished', '3':'Fully-Furnished'}
    furnishing = furn_map.get(f, 'Fully-Furnished')
    
    age_input = input("\nProperty Age in Years? [Press Enter for 2]: ").strip()
    property_age = int(age_input) if age_input else 2
    
    print("\nParking Options: [1] None, [2] Open, [3] Covered")
    p = input("Enter Parking (1-3)? [Press Enter for 3:Covered]: ").strip()
    park_map = {'1':'None', '2':'Open', '3':'Covered'}
    parking = park_map.get(p, 'Covered')
    
    print("\nRERA Approved Options: [1] Yes, [2] No")
    r = input("Enter RERA (1-2)? [Press Enter for 1:Yes]: ").strip()
    rera_map = {'1':'Yes', '2':'No'}
    rera_approved = rera_map.get(r, 'Yes')
    
    custom_house.at[0, 'City'] = city
    custom_house.at[0, 'Area_SqFt'] = area
    custom_house.at[0, 'BHK'] = bhk
    custom_house.at[0, 'Property_Age_Years'] = property_age
    custom_house.at[0, 'Furnishing'] = furnishing
    custom_house.at[0, 'Parking'] = parking
    custom_house.at[0, 'RERA_Approved'] = rera_approved
    
    pred_lakhs = loaded_model.predict(custom_house)[0]
    
    print("\n" + "*"*45)
    print(f" 🇮🇳 PREDICTED PRICE: ₹{pred_lakhs:,.2f} Lakhs ")
    if pred_lakhs > 100:
        crores = pred_lakhs / 100.0
        print(f" 🇮🇳 Or Approximately: ₹{crores:,.2f} Crores ")
    print("*"*45 + "\n")
except Exception as e:
    print(f"\n⚠️ An error occurred computing the prediction:\n{e}")
    


    🏠 ENTER CUSTOM HOUSE FEATURES 🏠    

City Options: [1] Mumbai, [2] Delhi, [3] Bangalore, [4] Hyderabad, [5] Chennai, [6] Pune, [7] Ahmedabad


Enter City Option (1-7)? [Press Enter for 3:Bangalore]:  1

Area (in SqFt)? [Press Enter for 1200]:  

BHK (1-6)? [Press Enter for 3]:  4



Furnishing Options: [1] Unfurnished, [2] Semi-Furnished, [3] Fully-Furnished


Enter Furnishing (1-3)? [Press Enter for 3:Fully-Furnished]:  3

Property Age in Years? [Press Enter for 2]:  10



Parking Options: [1] None, [2] Open, [3] Covered


Enter Parking (1-3)? [Press Enter for 3:Covered]:  3



RERA Approved Options: [1] Yes, [2] No


Enter RERA (1-2)? [Press Enter for 1:Yes]:  1



*********************************************
 🇮🇳 PREDICTED PRICE: ₹357.90 Lakhs 
 🇮🇳 Or Approximately: ₹3.58 Crores 
*********************************************



In [8]:
1


1